# Notebook 02: Evidence-Based Dataset Audit & Leakage Verification
Audits local or Drive-mounted datasets (ISLTranslate, iSign, ISL-CSLTR), checks video decoding integrity, verifies SHA-256 checksums, and tests for signer leakage.


In [ ]:
# 1. Setup workspace paths
import os, sys, glob, json
sys.path.insert(0, '.')
from ml.data.schemas import SignSample, SplitType
from ml.data.video_processor import VideoProcessor
from ml.data.manifest_generator import ManifestGenerator

DATA_DIR = 'data'
processor = VideoProcessor(target_fps=25)
print('Initialized video processor.')


In [ ]:
# 2. Scan raw videos and detect corrupt files
raw_videos = glob.glob(os.path.join(DATA_DIR, 'raw', '**', '*.mp4'), recursive=True)
print(f'Discovered {len(raw_videos)} raw video files.')
valid_videos = []
corrupt_videos = []

for v_path in raw_videos[:50]:  # Audit sample
    meta = processor.inspect_video(v_path)
    if meta.is_corrupt:
        corrupt_videos.append(v_path)
    else:
        valid_videos.append(meta)

print(f'Audited files -> Valid: {len(valid_videos)}, Corrupt: {len(corrupt_videos)}')


In [ ]:
# 3. Audit existing manifests and verify signer-disjoint partitions
manifest_path = os.path.join(DATA_DIR, 'manifests', 'train_manifest.json')
if os.path.exists(manifest_path):
    with open(manifest_path, 'r', encoding='utf-8') as f:
        train_data = json.load(f)
    with open(manifest_path.replace('train', 'val'), 'r', encoding='utf-8') as f:
        val_data = json.load(f)
    with open(manifest_path.replace('train', 'test'), 'r', encoding='utf-8') as f:
        test_data = json.load(f)

    passed, violations = ManifestGenerator.audit_leakage(
        [SignSample(**s) for s in train_data],
        [SignSample(**s) for s in val_data],
        [SignSample(**s) for s in test_data],
    )
    print('Signer and source leakage audit passed:', passed)
    if not passed:
        print('Violations detected:', violations)
else:
    print('No manifests found yet. Run Notebook 03 to generate manifests.')
